# Pasemos a cosas más grandes y más entrelazadas: Programación con múltiples qubits

## Resumen

Seguiremos utilizando la misma plantilla para crear un programa cuántico que en la primera parte:

* Codificar información en el estado cuántico inicializando uno o varios qubits
* Manipular el estado cuántico de los qubits con puertas cuánticas
* Extraer información del estado cuántico midiendo el estado de los qubits

Estos tres pasos se describen en el siguiente diagrama:

<img src="https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/main/images/circuit.png" alt="imagen de un circuito cuántico con las tres partes: codificar información, manipular estados cuánticos, extraer información" width="300"/>

Sin embargo, en esta sección, además de programar con más de un qubit, aprovecharemos esta oportunidad para demostrar una práctica de programación en la que se crean "subkernels" (similares a las subrutinas en un programa clásico) que se pueden reutilizar y combinar.

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import cmath
import cudaq
import qutip

# Programación con más de un qubit

En esta sección, crearemos dos programas cuánticos para ilustrar los conceptos de fase global y entrelazamiento. Además, estos ejemplos nos permitirán mostrar formas alternativas de escribir y combinar múltiples núcleos, una técnica útil que conviene tener a mano a medida que nuestros algoritmos cuánticos aumentan en número de qubits y de operaciones de puerta.  Tanto el número de qubits como el número de operaciones de puerta son factores limitantes en el tipo de algoritmos cuánticos que se pueden ejecutar en el hardware cuántico actual o simular en GPU.  De hecho, veremos rápidamente ejemplos de algoritmos cuánticos que podemos programar, pero que no podemos implementar con la tecnología existente hoy en día.

# Notación para un estado de 2 qubits
$\renewcommand{\ket}[1]{|#1\rangle}$
En el Laboratorio 1, vimos que el estado cuántico de un solo qubit podía escribirse como combinaciones lineales de los estados $|0\rangle$ y $|1\rangle$.  Los estados $|0\rangle$ y $|1\rangle$ se denominan a menudo estados de base computacional de un solo qubit.  Cuando tenemos 2 qubits, necesitaremos $2^2 = 4$ estados de base para describir todos los estados cuánticos posibles de 2 qubits, ya que cada qubit puede medirse como un $0$ o un $1$.  Los estados de base computacional utilizados para describir un estado de 2 qubits son
$$\ket{00}, \ket{01}, \ket{10},  \text{ y }\ket{11}.$$

¿Qué significa esta notación? La expresión $\ket{10}$ representa el estado de un sistema de 2 qubits (por ejemplo, $q_0$ y $q_1$), donde el qubit $q_0$ se encuentra en el estado $\ket{1}$ y $q_1$ se encuentra en el estado $\ket{0}$.  En otras palabras, el término más a la izquierda (el $1$ en $\ket{10}$) corresponde al estado del qubit $q_0$ y el término más a la derecha corresponderá al estado del segundo qubit.  Dado que no existe un orden estándar acordado para los qubits, esta elección de notación puede diferir de otros lenguajes de programación cuántica con los que te puedas encontrar.

La forma general de un estado cuántico de 2 qubits es una combinación lineal (o una superposición) de los estados de la base computacional:

$$\ket{\psi} = \alpha_{00}\ket{00}+\alpha_{10}\ket{10}+\alpha_{01}\ket{01}+\alpha_{11}\ket{11},$$
donde cada coeficiente (también llamado **amplitud de probabilidad**) $\alpha_{ij}$ es un número complejo y la suma de las probabilidades correspondientes a estos coeficientes es 1, es decir: $$\sum_{i,j\in\{0,1\}} |\alpha_{ij}|^2 =1.$$  

A veces resulta útil, aunque a menudo menos compacto, escribir el estado $\ket{\psi}$ como un vector de amplitudes de probabilidad:
$$\ket{\psi} = \alpha_{00}\ket{00}+\alpha_{10}\ket{10}+\alpha_{01}\ket{01}+\alpha_{11}\ket{11}
 = \begin{bmatrix}\alpha_{00} \\ \alpha_{01} \\ \alpha_{10} \\ \alpha_{11}\end{bmatrix}$$

> **Nota:** El orden en que aparecen los estados de la base computacional $(00, 10, 01, 11)$ en la suma difiere del orden en el vector vertical. Introducimos ambas convenciones porque el orden  $(00, 10, 01, 11)$ se utiliza cuando ejecutamos la función `get_state`, mientras que el orden del vector vertical $(00, 01, 10, 11)$ se utiliza en los cálculos de álgebra matricial y tensorial.

En la literatura, el término **vector de estado** se refiere a un estado cuántico $\ket{\psi}$, pero también puede referirse a la lista de amplitudes de probabilidad de un estado cuántico $\ket{\psi}$.  

# Escribir e interpretar un programa cuántico de 2 qubits

En esta sección, escribiremos un programa que se basa en los programas «Bit-Flip» y «Hello World» (estado negativo) del Laboratorio 1. En concreto, nuestro objetivo es crear un programa con dos qubits.  El primer qubit se inicializará en el estado cero y lo cambiaremos al estado uno. El segundo qubit se inicializará en el estado menos y, solo por diversión, también le aplicaremos la puerta de inversión de bits `x`. El diagrama representa el circuito de este programa.

<img src="https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/refs/heads/main/images/2-qubit-circuit.png" alt="quantum circuit diagram with 2 qubits" width="200"/>

En nuestro ejemplo, el qubit $q_0$, que comienza en el estado $\ket{0}$, sufre un cambio de estado. Por lo tanto, cabría esperar, tal como vimos en el Laboratorio 1, que cuando se muestree el núcleo en un entorno sin ruido, $q_0$ se mida como $\ket{1}$ el 100 % de las veces.   Por otro lado, dado que el qubit $q_1$ estará en un estado de superposición igual, colapsará al estado $\ket{0}$ aproximadamente la mitad de las veces y colapsará a $\ket{1}$ el resto de las veces. Al reunir la información sobre el estado de $q_0$ y $q_1$, esperamos una probabilidad del $50\%$ de muestrear un $\ket{10}$ y una probabilidad del $50\%$ de muestrear un $\ket{11}$ al ejecutar el programa descrito anteriormente.

El vector de estado del sistema de 2 qubits después de aplicar las puertas, pero antes de la medición, debería ser $$\ket{\psi} = -\frac{1}{\sqrt{2}}\ket{10} +\frac{1}{\sqrt{2}}\ket{11} = \begin{bmatrix}0 \\ 0 \\ -\frac{1}{\sqrt{2}} \\ \frac{1}{\sqrt{2}} \end{bmatrix}$$

Para entender la presencia del signo menos en la ecuación anterior, consulta la sección opcional sobre fase más adelante en este cuaderno. Por ahora, basta con observar que, al elevar al cuadrado los valores absolutos de las amplitudes, vemos que hay un 50 % de probabilidad de obtener un $\ket{10}$ y un 50 % de probabilidad de obtener un $\ket{11}$.

Tenemos varias opciones para escribir el código CUDA-Q que lleve a cabo las instrucciones anteriores. En las siguientes subsecciones, veremos dos opciones para que puedas apreciar algunas de las funcionalidades de `cudaq.kernel`. En particular, demostraremos
* cómo pasar qubits individuales o un registro de qubits a un kernel y
* cómo utilizar kernels como subkernels (subrutinas).

# Todo en un solo kernel

Primero programaremos el circuito cuántico como lo hicimos en el Laboratorio 1, escribiendo todas las instrucciones de forma secuencial en un solo kernel.

In [3]:
# Writing the all-in-one kernel

@cudaq.kernel
def all_in_one():
    """Creates a kernel with 2 qubits. The first qubit is initialized
    in the zero state, the second qubit is initialized in the minus state.
    An x gate is then applied to each qubit.
    """
    # Allocate 2 qubits (indexed by 0 and 1)
    qvector = cudaq.qvector(2) # By default these qubits are the zero state

    # Initialize qubit indexed with 1 as the minus state
    x(qvector[1])
    h(qvector[1])

    # Manipulate the quantum states by applying the bit flip gate (the `x` gate) to each qubit
    x(qvector) # We can apply a single-qubit gate to each qubit in a list of qubits

print(cudaq.draw(all_in_one))

     ╭───╮          
q0 : ┤ x ├──────────
     ├───┤╭───╮╭───╮
q1 : ┤ x ├┤ h ├┤ x ├
     ╰───╯╰───╯╰───╯



Comprobemos que este kernel genera nuestro estado objetivo $\ket{\psi} = -\frac{1}{\sqrt{2}}\ket{10} +\frac{1}{\sqrt{2}}\ket{11}$ utilizando el comando `get_state`.  También utilizaremos la opción `get_state.amplitudes` para extraer los coeficientes de los estados de la base computacional $\ket{10}$ y $\ket{11}$ que nos interesan.

In [4]:
# Compute the state of the system prior to measurement
all_in_one_state = cudaq.get_state(all_in_one)

# Return the amplitudes of |10> and |11>
all_in_one_amplitudes = all_in_one_state.amplitudes([[1,0], [1,1]])

# Print
precision = 4
print('All-in-one statevector array of coefficients:', np.round(np.array(all_in_one_state), precision))
print('All-in-one statevector: {} |10> + {} |11>'.format(np.round(all_in_one_amplitudes[0],precision), np.round(all_in_one_amplitudes[1],precision)))

All-in-one statevector array of coefficients: [ 0.    +0.j -0.7071+0.j  0.    +0.j  0.7071+0.j]
All-in-one statevector: (-0.7071+0j) |10> + (0.7071+0j) |11>


> **Consejo rápido:**

> Para recordar el orden de los coeficientes en el vector de estado generado por el comando `get_state`, resulta útil recordar la convención para asociar números con sus representaciones binarias, que coloca el bit menos significativo a la izquierda.  Por ejemplo, en un sistema de 2 bits, tenemos la siguiente correspondencia entre números enteros y bits:
$$\begin{matrix} \text{Número entero} & \text{Representación binaria}\\
& \text{bit menos significativo a la izquierda}\\
0 =\color{red}{0}*2^0+\color{blue}{0}*2^1 & \color{red}{0}\color{blue}{0} \\
1 = \color{red}{1}*2^0 + \color{blue}{0} *2^1 & \color{red}{1}\color{blue}{0}\\
2 = \color{red}{0}*2^0 + \color{blue}{1}*2^1 & \color{red}{0}\color{blue}{1} \\
3 = \color{red}{1}*2^0 + \color{blue}{1}*2^1 & \color{red}{1}\color{blue}{1} \end{matrix}
$$

> La salida de `get_state` genera una lista de los coeficientes en orden creciente: $\ket{00}, \ket{10}, \ket{01}, \ket{11}$, correspondientes a $0,1,2,3.$ Para identificar fácilmente el coeficiente de un estado de base dado, utilice el comando `get_state().amplitude()`. Por ejemplo, para extraer el coeficiente del estado de base $\ket{10}$, utilice `get_state(my_kernel).amplitude([1,0])`. Para recuperar las amplitudes de más de un estado a la vez, utilice la opción `amplitudes` aplicada a una lista de estados de base, tal como se muestra en el bloque de código anterior.

# Kernels como subrutinas

En esta sección, crearemos el mismo circuito cuántico que el anterior, pero esta vez utilizaremos kernels como subrutinas. Crearemos tres kernels distintos:

* `minus`, un kernel para la subrutina que inicializa un qubit dado en el estado «menos»
* `xgate`, un kernel que aplica la puerta `x` a una lista de qubits
* `nested_quantum_program`, un kernel para asignar los qubits y llamar a las subrutinas `minus` y `xgate`

Veamos cómo se hace esto. Fíjate en cómo podemos pasar qubits y listas de qubits a las funciones del kernel.

In [7]:
# Defining three kernels to generate the example circuit

# Defining the subroutine, minus, as a kernel
@cudaq.kernel
def minus(qubit: cudaq.qubit):
    """When applied to a qubit in the zero state, generates the minus state
    Parameters
    ----------
    qubit : cudaq.qubit
        qubit upon which the kernel instructions will be applied
    """
    x(qubit)
    h(qubit)

# Defining the subroutine, xgate, as a kernel
@cudaq.kernel
def xgate(qubits: cudaq.qvector):
    """Applies an x-gate to each qubit in qubits
    Parameters
    ----------
    qubits : cudaq.qvector
        list of qubits upon which the kernel instructions will be applied
    """
    x(qubits)

# Allocating qubits and calling the subroutines defined above
@cudaq.kernel
def nested_quantum_program(num_qubits: int):
    """Creates a kernel with num_qubits-many qubits. The first qubit is initialized
    in the zero state, the second qubit is initialized in the minus state.
    An x gate is then applied to each qubit. Here num_qubits is at least 2.
    Parameters
    ----------
    num_qubits : int
        Number of qubits to be allocated
    """

    # Allocate num_qubits qubits indexed from 0 to num_qubits-1
    qvector = cudaq.qvector(num_qubits) # By default these qubits are in the zero state

    # Initialize the qubit indexed with 1 as the minus state
    # by calling the minus kernel acting on qvector[1]
    minus(qvector[1])

    # Manipulate the quantum states by applying the bit flip gate (the `x` gate) to each qubit
    # by calling the xgate kernel applied to a list of qubits
    xgate(qvector)


num_qubits = 2 # num_qubits should be an integer > 1

print(cudaq.draw(nested_quantum_program,num_qubits))

     ╭───╮          
q0 : ┤ x ├──────────
     ├───┤╭───╮╭───╮
q1 : ┤ x ├┤ h ├┤ x ├
     ╰───╯╰───╯╰───╯



In [8]:
# EXERCISE 1
num_qubits = 2

# Compute the state of the system prior to measurement
state = cudaq.get_state(nested_quantum_program, num_qubits)
amplitudes = state.amplitudes([[1,0], [1,1]])
print('|Psi>= {} |10> + {} |11>'.format(np.round(amplitudes[0],precision), np.round(amplitudes[1],precision)))

|Psi>= (-0.7071+0j) |10> + (0.7071+0j) |11>


Ahora utilicemos el comando `sample` para comparar los dos programas cuánticos. Fíjate en la diferencia entre los argumentos del comando `sample` aplicado al núcleo `all_in_one` y los del
comando `sample` aplicado al núcleo `nested_quantum_program`, que toma como argumento `num_qubits`.

In [9]:
# Compare the quasi-probability distributions of each kernel upon sampling
shots = 1000

# Sample the kernels
counts_all_in_one = cudaq.sample(all_in_one, shots_count=shots)
counts_nested = cudaq.sample(nested_quantum_program, num_qubits, shots_count=shots)


# Print
print('All-in-one sampling results:', counts_all_in_one)
print('Nested results:', counts_nested)

All-in-one sampling results: { 10:502 11:498 }

Nested results: { 10:483 11:517 }



Antes de continuar, interpretemos el resultado del comando de ejemplo.
Según nuestra predicción, el resultado `{ 11:481 10:519 }` indica que el primer dígito de las cadenas de bits `11` y `10` corresponde al qubit $q_0$, ya que es `1` el 100 % de las veces, y el segundo dígito corresponde al qubit $q_1$, que se mide aproximadamente la mitad de las veces como un `0` y la otra mitad como un `1`. 

# Hello, Entangled World!

Hasta ahora, todas las operaciones cuánticas (por ejemplo, `x`, `h`) que hemos programado actuaban únicamente sobre un solo qubit. En esta sección, escribiremos un programa utilizando puertas de múltiples qubits. Comencemos con el ejemplo de la puerta «not» controlada, a menudo abreviada como puerta `CNOT`. Se trata de una puerta que opera sobre dos o más qubits. Uno de los qubits se considera el «objetivo», y los qubits restantes son el «control». Por ejemplo, `CNOT([q0,q1],q2)` representaría una puerta con qubits de control en la lista `[q0,q1]` y el qubit objetivo `q2`.  

En un diagrama de circuito, representamos la puerta `CNOT` como una línea que conecta los qubits de control y de destino con un símbolo $\bigoplus$ situado en el qubit de destino y un círculo sólido en los qubits de control. Por ejemplo, en el diagrama de circuito que se muestra a continuación, vemos una puerta `CNOT(q0, q1)` y una puerta `CNOT([q1,q2],q0)`.

<img src="https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/refs/heads/main/images/cnots.png" alt="quantum circuit diagram with various cnot gates" width="300"/>

La idea general es que, si todos los qubits de «control» se encuentran en el estado $\ket{1}$, la aplicación de la puerta `CNOT` provocará un cambio de bit (es decir, una puerta `x`) en el qubit de destino. Si, por el contrario, al menos uno de los qubits de «control» se encuentra en el estado $\ket{0}$, la puerta `CNOT` no tiene ningún efecto.

Para ilustrar esto, vamos a programar algunos ejemplos.

> **Consejo rápido:**
La sintaxis de la puerta CNOT es `x.ctrl(qubit_de_control, qubit_objetivo)` si solo tenemos un qubit de control.  Cuando tenemos varios qubits de control, los enviamos como una lista a la función `x.ctrl`: `x.ctrl(lista_de_qubits_de_control, qubit_objetivo)`.

In [10]:
# CNOT applied to the states |00>, |01>, |10>, |11>
print('In this example CNOT is applied with the control qubit q0 and the target q1.')

@cudaq.kernel
def apply_cnot(control: cudaq.qubit, target: cudaq.qubit):
    """Apply an CNOT gate to the control and target qubits"""
    x.ctrl(control, target)

@cudaq.kernel
def initialize(qubits: cudaq.qview, state: list[int]):
    """Kernel to initialize the state indicated by the given list
    0: |0>, 1: |1>"""
    for idx in range(len(state)):
        if state[idx] == 1:
            x(qubits[idx])

@cudaq.kernel
def CNOT_example(state: list[int]):
    """Apply CNOT to the state given by the state numbers"""
    qubits = cudaq.qvector(len(state))
    # Initialize state
    initialize(qubits, state)
    # Apply CNOT to the first two bits
    if len(state) > 1:
        apply_cnot(qubits[0], qubits[1])

# Test computational basis states
for i in range(4):
    bits = format(i,'b')
    bit_list = [int(bit) for bit in bits.zfill(2)]
    result = cudaq.sample(CNOT_example, bit_list).most_probable()
    print(f'CNOT applied to the state |{bits.zfill(2)}> results in |{result}>')

In this example CNOT is applied with the control qubit q0 and the target q1.
CNOT applied to the state |00> results in |00>
CNOT applied to the state |01> results in |01>
CNOT applied to the state |10> results in |11>
CNOT applied to the state |11> results in |10>


Si bien los ejemplos anteriores mostraban operaciones CNOT en estados básicos (|0⟩ y |1⟩), la herramienta [interactiva](https://nvidia.github.io/cuda-q-academic/quick-start-to-quantum/interactive_widget/cnot-visualization.html) que aparece a continuación te permite explorar comportamientos cuánticos más interesantes. Intenta seleccionar estados de superposición (|+⟩ o |-⟩) para uno o ambos qubits y observa cómo funciona la puerta CNOT. ¡Descubrirás algunos comportamientos cuánticos sorprendentes que no tienen equivalente clásico!  

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px;">
    <h3 style="color: #76b900; margin-top: 0;"> Ejercicio  3a:</h3>
    <p style="font-size: 16px; color: #333;">
Edita el bloque de código a continuación para ver el efecto de la secuencia de compuertas del diagrama siguiente sobre los qubits si $q_0$ se inicializó como $\ket{+}$ y $q_1$ se inicializó como $\ket{1}$
        <br><br>
<img src="https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/main/images/exercise3a.png" alt="image of a circuit with 3 alternating cnots and q_0 initialized as |+> and q_1 initialized as |1>" width="200"/>
    </p>
</div>

In [14]:
# EXERCISE 3 Part a
@cudaq.kernel
def alternating_cnots(qubit0: cudaq.qubit, qubit1: cudaq.qubit):
    """Apply a sequence of 3 CNOTs with alternating controls and targets on the 2 qubits given as arguments"""
    # We will see later that it doesn't matter which qubit you start out with as the first control, as long as
    # you alternate the control and target qubits with each of the 3 applications of the cnot gate

    # Edit code below this line
    x.ctrl(qubit0, qubit1)

    # Edit code above this line
    x.ctrl(qubit1, qubit0)

    x.ctrl(qubit0, qubit1)


@cudaq.kernel
def three_alternating_cnots():
    """Kernel for the circuit drawn above"""

    # Allocate qubits
    q = cudaq.qvector(2)

    # Initialize qubits q0 and q1 in the plus and one states, respectively
    h(q[0])
    x(q[1])

    # Apply alternating CNOTs
    alternating_cnots(q[0], q[1])



results = cudaq.sample(three_alternating_cnots)
print('The distribution of states after sampling is: {}'.format(results))

The distribution of states after sampling is: { 10:492 11:508 }



¿Te diste cuenta de que el kernel `alternating_cnots` tenía el efecto de intercambiar los estados de $q_0$ y $q_1$?  Esta es una operación útil, por lo que existe una notación específica para ella.  Nos referimos a esta secuencia de puertas como la puerta `SWAP`, y la abreviamos como una línea con dos X en los extremos que conectan los qubits:

<img src="https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/main/images/SWAP.png" alt="image of a circuit with a swap gate and one of three alternating cnots" width="200"/>

El comando CUDA-Q para una puerta SWAP aplicada a los qubits $q_0$ y $q_1$ es `swap(q_0, q_1)`. Usemos CUDA-Q para comprobar que la puerta `swap` tiene el mismo efecto sobre dos qubits que el núcleo `alternating_cnots`.

In [12]:
# Verify that the SWAP and alternating_cnots have the same effect on the state |+1>

@cudaq.kernel
def applying_swap():
    """Kernel for the circuit drawn above"""

    # Allocate qubits
    q = cudaq.qvector(2)

    # Initialize qubits q0 and q1 in the plus and one states, respectively
    h(q[0])
    x(q[1])

    # Apply alternating CNOTs
    swap(q[0], q[1])

results = cudaq.sample(three_alternating_cnots)
print('The distribution of states after sampling is: {}'.format(results))

The distribution of states after sampling is: { 10:494 11:506 }



Ahora, veamos otro ejemplo que nos mostrará uno de los potentes cálculos que se pueden realizar con puertas cuánticas. Aplica la puerta `x.ctrl` al estado $\ket{+0}$.

In [16]:
# EXERCISE 3 Part b

@cudaq.kernel
def apply_cnot(control: cudaq.qubit, target: cudaq.qubit):
    """Apply an CNOT gate to the control and target qubits"""
    x.ctrl(control, target)

@cudaq.kernel
def initialize_plus_zero(qubits: cudaq.qview):
    """Kernel to initialize the state indicated by the given list of bits"""
    # Place qubits[0] in the plus state
    h(qubits[0])

@cudaq.kernel
def CNOT_exercise():
    """Apply CNOT to |+0> with control q0 and target q1"""
    qubits =cudaq.qvector(2)
    # Initialize state
    initialize_plus_zero(qubits)
    # Apply CNOT to the first two bits
    apply_cnot(qubits[0], qubits[1])

results = cudaq.sample(CNOT_exercise)
print('CNOT applied to the state |+0> results in the distribution of states: {}'.format(results))


CNOT applied to the state |+0> results in the distribution of states: { 00:518 11:482 }



Para entender el resultado del ejercicio anterior, utilicemos la notación ket. En primer lugar, debemos señalar que una característica de las compuertas cuánticas que no hemos mencionado explícitamente es la linealidad.  Por ejemplo, si tenemos una puerta $U$ aplicada al estado $\ket{\psi}=\alpha\ket{0}+\beta\ket{1}$, podemos entender la acción de $U$ sobre $\ket{\psi}$ sabiendo cómo $U$ afecta a $\ket{0}$ y $\ket{1}$.  En particular, podemos distribuir $U$ entre los términos de $\ket{\psi}$:
$$ U\ket{\psi} = U(\alpha\ket{0}+\beta\ket{1}) =  U(\alpha\ket{0}) +  U(\beta\ket{1})=\alpha U\ket{0} + \beta U\ket{1}.$$
Esto se generaliza más allá de las puertas de un qubit a las puertas de múltiples qubits.

Por lo tanto, cuando aplicamos el `CNOT` a $\ket{+0} = \frac{1}{\sqrt{2}}\ket{00}+ \frac{1}{\sqrt{2}}\ket{10}$ obtenemos

$$CNOT\ket{+0} = CNOT(\frac{1}{\sqrt{2}}\ket{00}+ \frac{1}{\sqrt{2}}\ket{10})=\frac{1}{\sqrt{2}}CNOT\ket{00} + \frac{1}{\sqrt{2}}CNOT\ket{10} = \frac{1}{\sqrt{2}}\ket{00} +  \frac{1}{\sqrt{2}}\ket{11}.$$

# Las puertas de múltiples qubits como matrices

Para comprender por qué las operaciones de las puertas son lineales, resulta útil representar una puerta de múltiples qubits como una matriz, tal como lo hicimos en el Laboratorio 1 con las puertas de un solo qubit.

Por ejemplo, la matriz para la puerta CNOT utilizando la convención de ordenación CUDA-Q es:

$$CNOT = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{bmatrix}.
$$

Utilizando el hecho de que  $\ket{+0} = \frac{1}{\sqrt{2}}\ket{00}+ \frac{1}{\sqrt{2}}\ket{10}$, podemos verificar nuestro trabajo de la sección anterior:

$$ CNOT\ket{+0} = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{bmatrix}(\frac{1}{\sqrt{2}}\ket{00}+ \frac{1}{\sqrt{2}}\ket{10})
=  \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{bmatrix}
\begin{bmatrix} \frac{1}{\sqrt{2}}  \\ 0 \\ \frac{1}{\sqrt{2}} \\ 0\end{bmatrix} =
\begin{bmatrix} \frac{1}{\sqrt{2}} \\ 0 \\ 0 \\ \frac{1}{\sqrt{2}}  \end{bmatrix} =
 \frac{1}{\sqrt{2}}\ket{00} +  \frac{1}{\sqrt{2}}\ket{11}
$$

El estado $\frac{1}{\sqrt{2}}\ket{00} +  \frac{1}{\sqrt{2}}\ket{11}$ es un ejemplo de entrelazamiento. Junto con la superposición, esta propiedad ayuda a distinguir los algoritmos cuánticos de los clásicos. Veamos ahora esta propiedad.

> **Preguntas frecuentes:** Si los estados cuánticos pueden representarse como vectores y las puertas cuánticas pueden realizarse mediante operaciones matriciales, ¿por qué necesitamos computadoras cuánticas si, en teoría, podemos simular cualquier circuito cuántico mediante la multiplicación de matrices de forma clásica?

> **Respuesta:**  Una cosa a tener en cuenta es que, a medida que aumenta el número de qubits, el número de estados posibles aumenta exponencialmente y el tamaño de la matriz que representa la puerta de múltiples qubits también aumenta exponencialmente.  Por ejemplo, un sistema de 3 qubits (como el del Ejercicio 4 de la siguiente sección) puede encontrarse en uno de los $2^3 = 8$ estados, y el tamaño de la matriz que representa una puerta de 3 qubits es de $8\times 8$.  Esto sugiere que los intentos directos de utilizar la multiplicación de matrices para simular el cálculo cuántico en CPU fracasan para circuitos con tan solo unos 30 qubits.  Con la plataforma de CUDA-Q y los simuladores integrados acelerados por GPU, es posible simular sistemas más grandes. Dicho esto, es posible que algún día exista una computadora cuántica tolerante a fallos capaz de realizar cálculos demasiado grandes incluso para las supercomputadoras clásicas más grandes.

# Entrelazamiento

El entrelazamiento de dos (o más) qubits es similar al concepto de dependencia en álgebra lineal. Es decir, dos qubits están entrelazados si el estado de uno de ellos depende del otro. Formalmente, la definición de entrelazamiento se basa en productos tensoriales, que quedan fuera del alcance de esta serie de cuadernos de «Inicio rápido».  Puedes leer más sobre los productos tensoriales y la definición precisa de entrelazamiento [aquí](http://mermin.lassp.cornell.edu/qcomp/CS483.html) o [aquí](https://qubit.guide).  Por ahora, ilustraremos el entrelazamiento mediante el ejemplo del estado de 2 qubits

$$ \ket{\psi} =\frac{1}{\sqrt{2}}\ket{00} +  \frac{1}{\sqrt{2}}\ket{11}.$$

El estado $\ket{\psi}$ tiene la misma probabilidad de ser medido en el estado $\ket{00}$ o en el estado $\ket{11}$. Veamos qué sucede si medimos el primer qubit.

* Si nuestro sistema de dos qubits se encuentra en el estado $\ket{\psi}$ y medimos que el qubit 0 es $0$, entonces el estado del sistema colapsa a $\ket{00}$.  En este caso, sin realizar una medición en el qubit 1, ya conocemos su estado (es $\ket{0}$).

<img src="https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/main/images/bell0.png" alt="image of a circuit for the Bell State with the first qubit measured to be a 0" width="300"/>

* Del mismo modo, si el estado de nuestro sistema es $\ket{\psi}$ y se mide que el qubit 0 es 1, entonces el estado del sistema colapsa a $\ket{11}$.  Aunque antes de cualquier medición del qubit 0, el qubit 1 tenía un 50 % de probabilidad de estar en el estado $\ket{0}$ o $\ket{1}$, tras medir que el qubit 0 es 1, sabemos que el qubit 1 debe estar en el estado $\ket{1}$.

<img src="https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/main/images/bell1.png" alt="image of a circuit for the Bell State with the first qubit measured to be a 1" width="300"/>

Así pues, vemos que, aunque antes de la medición tanto el qubit 0 como el qubit 1 tienen la misma probabilidad de dar un 0 o un 1, una vez que medimos uno de estos qubits, el resultado del otro qubit queda determinado y depende del estado del qubit que se ha medido. En otras palabras, el qubit 0 y el qubit 1 están **entrelazados**.